<a href="https://colab.research.google.com/github/ayush121325/Agro-weather-AI/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/Avishkar_Hackathon/processed"

train = pd.read_csv(f"{OUT_DIR}/clean_train.csv")
test = pd.read_csv(f"{OUT_DIR}/clean_test.csv")
rainfall = pd.read_csv(f"{OUT_DIR}/clean_rainfall.csv")
soil = pd.read_csv(f"{OUT_DIR}/clean_soil.csv")

# Helper: normalize State names to lowercase so merges always match
def norm_state(s):
    s = str(s).strip().lower().replace("&", "and")
    fixes = {
        "nct of delhi": "delhi",
        "jammu and kashmir": "jammu and kashmir",
        "andaman and nicobar": "andaman and nicobar islands",
        "orissa": "odisha"
    }
    return fixes.get(s, s)

train["state_key"] = train["State"].apply(norm_state)
test["state_key"] = test["State"].apply(norm_state)
rainfall["state_key"] = rainfall["state"].apply(norm_state)

Mounted at /content/drive


In [ ]:
# 1. Per-hectare fertilizer and pesticide
for df in [train, test]:
    df["Fertilizer_per_ha"] = df["Fertilizer"] / df["Area"].clip(lower=1e-6)
    df["Pesticide_per_ha"] = df["Pesticide"] / df["Area"].clip(lower=1e-6)

# 2. Long-Period Average (LPA) annual rainfall per state from TRAIN ONLY
state_lpa = train.groupby("state_key")["Annual_Rainfall"].mean().to_dict()
global_lpa = train["Annual_Rainfall"].mean()

for df in [train, test]:
    df["LPA_Rainfall"] = df["state_key"].map(state_lpa).fillna(global_lpa)
    # Formula: (Actual - Normal) / Normal * 100
    df["Rainfall_Deviation_pct"] = (
        (df["Annual_Rainfall"] - df["LPA_Rainfall"]) / df["LPA_Rainfall"]
    ) * 100.0

print(train[["State", "Crop_Year", "Annual_Rainfall", "LPA_Rainfall", "Rainfall_Deviation_pct"]].head())

         State  Crop_Year  Annual_Rainfall  LPA_Rainfall  \
0        Assam       1997           2051.4   2081.349870   
1    Karnataka       1997           1266.7   1156.693203   
2       Kerala       1997           3252.4   2858.825991   
3    Meghalaya       1997           3818.2   3959.484354   
4  West Bengal       1997           1852.9   1707.210231   

   Rainfall_Deviation_pct  
0               -1.438964  
1                9.510456  
2               13.766980  
3               -3.568251  
4                8.533792  


In [ ]:
day_cols = [c for c in rainfall.columns if c not in ["state", "district", "month", "state_key"]]

# Helper to find longest streak of dry days (< 2.5 mm) in a month
def max_dry_streak(row_vals):
    max_streak, cur = 0, 0
    for val in row_vals:
        if val < 2.5:
            cur += 1
            max_streak = max(max_streak, cur)
        else:
            cur = 0
    return max_streak

daily_matrix = rainfall[day_cols].values
rainfall["monthly_rain_mm"] = daily_matrix.sum(axis=1)
rainfall["rainy_days"] = (daily_matrix >= 2.5).sum(axis=1)
rainfall["max_dry_spell_month"] = [max_dry_streak(r) for r in daily_matrix]

# Map Indian seasons to their months
season_months = {
    "Kharif": [6, 7, 8, 9, 10],
    "Rabi": [11, 12, 1, 2, 3],
    "Summer": [4, 5],
    "Autumn": [9, 10, 11],
    "Winter": [12, 1, 2],
    "Whole Year": list(range(1, 13))
}

# Build a State + Season climate table
seasonal_rows = []
for season_name, months in season_months.items():
    sub = rainfall[rainfall["month"].isin(months)]
    agg = sub.groupby("state_key").agg(
        season_avg_rain_mm=("monthly_rain_mm", "mean"),
        season_rainy_days=("rainy_days", "mean"),
        season_max_dry_spell=("max_dry_spell_month", "max"),
        season_rain_std=("monthly_rain_mm", "std")
    ).reset_index()
    agg["Season"] = season_name
    seasonal_rows.append(agg)

season_climate = pd.concat(seasonal_rows, ignore_index=True)

# Merge onto train and test
train = train.merge(season_climate, on=["state_key", "Season"], how="left")
test = test.merge(season_climate, on=["state_key", "Season"], how="left")

# Fill any unmatched state-seasons with train medians
for col in ["season_avg_rain_mm", "season_rainy_days", "season_max_dry_spell", "season_rain_std"]:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

print("Merged rainfall timing features successfully!")

Merged rainfall timing features successfully!


In [ ]:
# 1. Summarize optimal soil & climate conditions per crop from soil_recommendation.csv
soil_profile = soil.groupby("label").agg(
    opt_N=("N", "mean"),
    opt_P=("P", "mean"),
    opt_K=("K", "mean"),
    opt_ph=("ph", "mean"),
    opt_temp=("temperature", "mean"),
    opt_humidity=("humidity", "mean"),
    opt_rainfall=("rainfall", "mean")
).reset_index()

# Map crop names from train/test to soil_recommendation labels
crop_to_soil = {
    "rice": "rice", "maize": "maize", "gram": "chickpea",
    "arhar/tur": "pigeonpeas", "moong(green gram)": "mungbean",
    "urad": "blackgram", "masoor": "lentil", "moth": "mothbeans",
    "banana": "banana", "coconut": "coconut", "cotton(lint)": "cotton",
    "jute": "jute", "coffee": "coffee"
}

for df in [train, test]:
    df["soil_label"] = df["Crop"].str.lower().map(crop_to_soil)

train = train.merge(soil_profile, left_on="soil_label", right_on="label", how="left").drop(columns=["label"])
test = test.merge(soil_profile, left_on="soil_label", right_on="label", how="left").drop(columns=["label"])

# Fill crops not in the 22-crop soil file with overall soil medians
soil_cols = ["opt_N", "opt_P", "opt_K", "opt_ph", "opt_temp", "opt_humidity", "opt_rainfall"]
for col in soil_cols:
    med = soil_profile[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

# 2. Soil & Climate Adequacy Index (0 to 1 scale: closer to 1 = closer to crop's ideal conditions)
for df in [train, test]:
    # Compare seasonal average rainfall against crop's optimal monthly rainfall
    rain_fit = 1.0 - (np.abs(df["season_avg_rain_mm"] - df["opt_rainfall"]) / (df["opt_rainfall"] + 100.0)).clip(0, 1)
    # Compare fertilizer intensity against crop's optimal N+P+K target
    npk_target = df["opt_N"] + df["opt_P"] + df["opt_K"]
    fert_fit = 1.0 - (np.abs(df["Fertilizer_per_ha"] - npk_target) / (npk_target + 150.0)).clip(0, 1)
    df["Soil_Adequacy_Index"] = (0.5 * rain_fit + 0.5 * fert_fit).round(4)

# 3. GDD (Growing Degree Days)
# Exact agronomic formula for when daily Tmax & Tmin are available:
def calculate_daily_gdd(tmax_list, tmin_list, t_base=10.0):
    return sum(max(0.0, ((hi + lo) / 2.0) - t_base) for hi, lo in zip(tmax_list, tmin_list))

# Crop-specific base temperature (deg C) and approximate growing season days
tbase_map = {"wheat": 4.4, "barley": 4.4, "potato": 7.0, "rice": 10.0, "maize": 10.0, "sugarcane": 12.0}
season_days_map = {"Kharif": 120, "Rabi": 120, "Summer": 90, "Autumn": 100, "Winter": 100, "Whole Year": 300}

for df in [train, test]:
    t_base = df["Crop"].str.lower().map(tbase_map).fillna(10.0)
    days = df["Season"].map(season_days_map).fillna(120)
    # Proxy GDD derived from available crop-level optimal temperature
    df["Proxy_GDD"] = np.maximum(0.0, df["opt_temp"] - t_base) * days

print("Soil Adequacy & Proxy GDD created!")
print(train[["Crop", "Season", "Soil_Adequacy_Index", "Proxy_GDD"]].head())

Soil Adequacy & Proxy GDD created!
       Crop      Season  Soil_Adequacy_Index    Proxy_GDD
0  Arecanut  Whole Year               0.7182  4574.827897
1  Arecanut  Whole Year               0.9344  4574.827897
2  Arecanut  Whole Year               0.7095  4574.827897
3  Arecanut  Whole Year               0.4721  4574.827897
4  Arecanut  Whole Year               0.8752  4574.827897


In [ ]:
# Sort train chronologically before creating lagged yield
train = train.sort_values(["State", "Crop", "Season", "Crop_Year"]).reset_index(drop=True)

# Previous season's yield in train (shifted by 1 year within each group)
train["prev_year_yield"] = train.groupby(["State", "Crop", "Season"])["Yield"].shift(1)

# Fallbacks if it's the very first year of a crop-state pair
crop_state_mean = train.groupby(["State", "Crop"])["Yield"].mean()
crop_mean = train.groupby("Crop")["Yield"].mean()
global_yield_mean = train["Yield"].mean()

def fill_lag_train(row):
    if pd.notna(row["prev_year_yield"]):
        return row["prev_year_yield"]
    return crop_state_mean.get((row["State"], row["Crop"]), crop_mean.get(row["Crop"], global_yield_mean))

train["prev_year_yield"] = train.apply(fill_lag_train, axis=1)

# For TEST (2018-2020), use the MOST RECENT known yield from train (up to 2017)
latest_train_yield = (
    train.sort_values("Crop_Year")
    .groupby(["State", "Crop", "Season"])["Yield"]
    .last()
)

def get_lag_test(row):
    key = (row["State"], row["Crop"], row["Season"])
    if key in latest_train_yield.index:
        return latest_train_yield.loc[key]
    return crop_state_mean.get((row["State"], row["Crop"]), crop_mean.get(row["Crop"], global_yield_mean))

test["prev_year_yield"] = test.apply(get_lag_test, axis=1)

# Drop temporary helper columns and save final datasets for Member 2
drop_helpers = ["state_key", "soil_label"]
train.drop(columns=drop_helpers).to_csv(f"{OUT_DIR}/features_train.csv", index=False)
test.drop(columns=drop_helpers).to_csv(f"{OUT_DIR}/features_test.csv", index=False)

print("DONE! Hand these two files to Member 2:")
print(" - features_train.csv:", train.shape)
print(" - features_test.csv:", test.shape)

DONE! Hand these two files to Member 2:
 - features_train.csv: (17305, 30)
 - features_test.csv: (2179, 29)
